# 06. Train and deploy a shipping-cost model

## Business use

Checkout asks for a shipping quote. The answer is a distribution: expected cost, median, a 90% predictive interval, and the probability the cost exceeds the price we already quoted. Carrier and zone are partially pooled. Metro is the baseline zone.

## Production translation

The model does not own the price the customer sees. It returns a distribution. Pricing owns floors, margin rules, and rounding.

| Pattern | Use when | Example |
|---|---|---|
| Batch scores in a table | The output changes slowly and many consumers read it | Nightly segment and lifecycle state |
| Online REST API | The input arrives per request and there is a latency budget | This shipping quote |
| Event consumer | State changes with each event | Session events update an HMM probability vector |
| Precomputed cache | Inputs are a small grid | Price response per SKU, refreshed daily |

For a service that is not Python, the hand-off is the OpenAPI schema at `/openapi.json`. Clients are generated from it, so the contract is checked on both sides. Version the path (`/v1/...`) separately from `model_version`.

## Model

Log cost is normal. Carrier intercepts share a common mean. Zone effects are differences from metro.

$$
\log \mathrm{cost} = a_{\mathrm{carrier}} + \delta_{\mathrm{zone}} + b_w \log(\mathrm{weight}) + b_d \log(\mathrm{distance}/250) + \varepsilon
$$

| Parameter | Prior |
|---|---|
| $\mu_a$, $\sigma_a$ | Normal(1.5, 1), HalfNormal(0.5) |
| $a_c$ | Normal($\mu_a$, $\sigma_a$) |
| zone raw | Normal(0, 0.5), then subtract the metro value |
| $b_w$, $b_d$ | Normal(0.5, 0.3), Normal(0.2, 0.3) |
| $\sigma$ | HalfNormal(0.5) |

Fit offline. Keep 1,000 thinned draws in `posterior_draws.npz`. The API does arithmetic with those draws. It does not call the sampler. The full InferenceData, including replicated costs, is the audit file. On this model it is about 100 MB, so the training script writes it, prints the size, and deletes it. Regenerate with `python train.py` if you need the netcdf locally. Do not commit it.

Gates before export: zero divergences, R-hat below 1.01, bulk ESS above 400. The fixed metro effect has R-hat `nan`. That parameter is a constant zero. Drop it before the max.


## Step 1. Simulated shipments


In [1]:
import importlib
import inspect
import json
import shutil
import signal
import subprocess
import sys
import time
from pathlib import Path

import httpx
import numpy as np

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DEPLOY = ROOT / "deploy"
sys.path.insert(0, str(DEPLOY))
train = importlib.import_module("train")

shipments = train.simulate(4000, np.random.default_rng(2026))
print("rows:", len(shipments), "columns:", list(shipments.columns))
print(shipments.dtypes.astype(str).to_dict())
print(shipments.describe().round(2).to_string())

rows: 4000 columns: ['carrier', 'zone', 'weight_kg', 'distance_km', 'cost_usd']
{'carrier': 'str', 'zone': 'str', 'weight_kg': 'float64', 'distance_km': 'float64', 'cost_usd': 'float64'}
       weight_kg  distance_km  cost_usd
count    4000.00      4000.00   4000.00
mean        1.89       310.06      8.94
std         1.75       250.68      6.43
min         0.06        19.90      0.93
25%         0.78       150.25      4.65
50%         1.38       244.45      7.13
75%         2.38       389.28     11.20
max        24.45      3188.30     91.39


## Step 2. Fit, gate, export

The cell prints `deploy/train.py` and then runs it. Four chains, 1,000 tuning steps, 1,000 draws, `target_accept=0.9`, seed 1.


In [2]:
print(inspect.getsource(train.fit_and_export))
train.fit_and_export(shipments)

Initializing NUTS using jitter+adapt_diag...


def fit_and_export(shipments: pd.DataFrame, out_dir: Path = ROOT) -> None:
    carrier_index = shipments["carrier"].map({name: i for i, name in enumerate(CARRIERS)}).to_numpy()
    zone_index = shipments["zone"].map({name: i for i, name in enumerate(ZONES)}).to_numpy()
    log_weight = np.log(shipments["weight_kg"].to_numpy())
    log_distance = np.log(shipments["distance_km"].to_numpy() / 250)
    log_cost = np.log(shipments["cost_usd"].to_numpy())

    with pm.Model(coords={"carrier": CARRIERS, "zone": ZONES}):
        mu_a = pm.Normal("mu_a", 1.5, 1.0)
        sd_a = pm.HalfNormal("sd_a", 0.5)
        intercept = pm.Normal("a", mu_a, sd_a, dims="carrier")
        zone_raw = pm.Normal("z_raw", 0, 0.5, dims="zone")
        # Metro is the baseline, so zone effects are differences from zone 0.
        zone_effect = pm.Deterministic("zone_eff", zone_raw - zone_raw[0], dims="zone")
        weight_slope = pm.Normal("b_w", 0.5, 0.3)
        distance_slope = pm.Normal("b_d", 0.2, 0.3)
      

Multiprocess sampling (4 chains in 2 jobs)


NUTS: [mu_a, sd_a, a, z_raw, b_w, b_d, sigma]


/workspace/bayesian-modelling/.venv/lib/python3.12/site-packages/pymc/step_methods/hmc/quadpotential.py:321: RuntimeWarning: overflow encountered in dot
  return 0.5 * np.dot(x, v_out)


Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 49 seconds.


Sampling: [log_cost]


/workspace/bayesian-modelling/.venv/lib/python3.12/site-packages/arviz_stats/base/diagnostics.py:90: RuntimeWarning: invalid value encountered in scalar divide
  (between_chain_variance / within_chain_variance + num_samples - 1) / (num_samples)
/workspace/bayesian-modelling/.venv/lib/python3.12/site-packages/arviz_stats/base/diagnostics.py:313: RuntimeWarning: invalid value encountered in scalar divide
  varsd = varvar / evar / 4


                       mean       sd eti89_lb eti89_ub ess_bulk ess_tail r_hat mcse_mean  mcse_sd
a[carrier_a]         1.5924   0.0098      1.6      1.6     3918     2975  1.00   0.00016  0.00015
a[carrier_b]          1.389   0.0097      1.4      1.4     4234     3358  1.00   0.00015  0.00014
a[carrier_c]         1.1882     0.01      1.2      1.2     3734     2926  1.00   0.00016  0.00016
a[carrier_d]         2.1029   0.0098      2.1      2.1     4064     3227  1.00   0.00015  0.00014
zone_eff[metro]           0        0        0        0     4000     4000   nan         0      nan
zone_eff[regional]   0.2501   0.0096     0.23     0.27     3413     3303  1.00   0.00016  0.00013
zone_eff[remote]     0.6094   0.0098     0.59     0.63     3302     3356  1.00   0.00017  0.00013
b_w                  0.4559   0.0048     0.45     0.46     4559     2766  1.00   7.2e-05  8.6e-05
b_d                  0.2014   0.0057     0.19     0.21     5001     2755  1.00   8.1e-05   0.0001
sigma               

PPC 90% interval coverage (train): 0.898
{"model_name": "shipping_cost_hier_lognormal", "model_version": "1.0.0", "train_data_sha": "9b58a477cf94", "carriers": ["carrier_a", "carrier_b", "carrier_c", "carrier_d"], "zones": ["metro", "regional", "remote"], "n_draws": 1000, "max_rhat": 1.0028984449415725, "min_ess_bulk": 3302}
serving shapes: {'a': (4, 1000), 'zone_eff': (3, 1000), 'b_w': (1000,), 'b_d': (1000,), 'sigma': (1000,)}
audit netcdf bytes (deleted, not committed): 129160662


## Step 3. The serving calculation, without HTTP

Index the carrier and the zone, add the slopes times log weight and log distance, then draw observation noise. The mean sits above the median because the cost is log-normal. `out_of_training_range` is true when log weight is outside the training min and max.


In [3]:
meta = json.loads((DEPLOY / "model_meta.json").read_text(encoding="utf-8"))
draws = np.load(DEPLOY / "posterior_draws.npz")
print("serving shapes:", {name: draws[name].shape for name in draws.files})
print("max rhat:", round(meta["max_rhat"], 4), "min ess:", meta["min_ess_bulk"])

carrier_index = {name: i for i, name in enumerate(meta["carriers"])}
zone_index = {name: i for i, name in enumerate(meta["zones"])}
weight_kg, distance_km, quoted = 2.5, 600.0, 9.5
mean = (
    draws["a"][carrier_index["carrier_b"]]
    + draws["zone_eff"][zone_index["regional"]]
    + draws["b_w"] * np.log(weight_kg)
    + draws["b_d"] * np.log(distance_km / 250)
)
predictive = np.exp(np.random.default_rng(0).normal(mean, draws["sigma"]))
quantiles = meta["train_ref"]["log_weight_quantiles"]
local_quote = {
    "expected_cost_usd": round(float(predictive.mean()), 2),
    "median_cost_usd": round(float(np.median(predictive)), 2),
    "ci90_low_usd": round(float(np.quantile(predictive, 0.05)), 2),
    "ci90_high_usd": round(float(np.quantile(predictive, 0.95)), 2),
    "prob_cost_exceeds_quote": round(float(np.mean(predictive > quoted)), 3),
    "out_of_training_range": not (quantiles[0] <= np.log(weight_kg) <= quantiles[-1]),
}
print(local_quote)
assert 1 < local_quote["expected_cost_usd"] < 40
assert local_quote["median_cost_usd"] <= local_quote["expected_cost_usd"]
assert local_quote["out_of_training_range"] is False

serving shapes: {'a': (4, 1000), 'zone_eff': (3, 1000), 'b_w': (1000,), 'b_d': (1000,), 'sigma': (1000,)}
max rhat: 1.0029 min ess: 3302


{'expected_cost_usd': 9.49, 'median_cost_usd': 9.17, 'ci90_low_usd': 6.33, 'ci90_high_usd': 13.58, 'prob_cost_exceeds_quote': 0.444, 'out_of_training_range': False}


## Step 4. Run the FastAPI app with uvicorn

Same process model as production: `uvicorn app:app`. Requests: health, a normal quote, a heavy out-of-range parcel, and an invalid body. The invalid body must be HTTP 422 from the pydantic contract, not a stack trace.


In [4]:
print("docker on PATH:", shutil.which("docker"))
process = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app:app", "--host", "127.0.0.1", "--port", "8765"],
    cwd=DEPLOY,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.PIPE,
    text=True,
)
base = "http://127.0.0.1:8765"
try:
    health = None
    for _ in range(50):
        try:
            health = httpx.get(base + "/health", timeout=0.2)
            if health.status_code == 200:
                break
        except httpx.HTTPError:
            time.sleep(0.1)
    else:
        error = process.stderr.read() if process.stderr is not None else ""
        raise RuntimeError(error)
    print("GET /health", health.json())

    normal = httpx.post(
        base + "/v1/shipping-cost/quote",
        json={
            "carrier": "carrier_b",
            "zone": "regional",
            "weight_kg": 2.5,
            "distance_km": 600,
            "quoted_price_usd": 9.5,
        },
        timeout=5,
    )
    heavy = httpx.post(
        base + "/v1/shipping-cost/quote",
        json={"carrier": "carrier_d", "zone": "remote", "weight_kg": 45, "distance_km": 1800},
        timeout=5,
    )
    invalid = httpx.post(
        base + "/v1/shipping-cost/quote",
        json={"carrier": "carrier_x", "zone": "regional", "weight_kg": -1, "distance_km": 600},
        timeout=5,
    )
    print("POST normal", normal.status_code, normal.json())
    print("POST heavy", heavy.status_code, heavy.json())
    print("POST invalid", invalid.status_code, invalid.json())
    schema = httpx.get(base + "/openapi.json", timeout=5).json()
    print("openapi paths:", sorted(schema["paths"]))
    assert normal.status_code == 200
    assert heavy.status_code == 200
    assert invalid.status_code == 422
    assert normal.json()["model_version"] == "1.0.0"
    assert heavy.json()["out_of_training_range"] is True
    assert normal.json()["expected_cost_usd"] == local_quote["expected_cost_usd"]
finally:
    process.send_signal(signal.SIGTERM)
    process.wait(timeout=10)

docker on PATH: None


GET /health {'status': 'ok', 'model_version': '1.0.0', 'train_data_sha': '9b58a477cf94'}
POST normal 200 {'model_name': 'shipping_cost_hier_lognormal', 'model_version': '1.0.0', 'expected_cost_usd': 9.49, 'median_cost_usd': 9.17, 'ci90_low_usd': 6.33, 'ci90_high_usd': 13.58, 'prob_cost_exceeds_quote': 0.444, 'out_of_training_range': False, 'latency_ms': 6.85}
POST heavy 200 {'model_name': 'shipping_cost_hier_lognormal', 'model_version': '1.0.0', 'expected_cost_usd': 131.1, 'median_cost_usd': 127.3, 'ci90_low_usd': 82.97, 'ci90_high_usd': 190.9, 'prob_cost_exceeds_quote': None, 'out_of_training_range': True, 'latency_ms': 0.74}
POST invalid 422 {'detail': [{'type': 'literal_error', 'loc': ['body', 'carrier'], 'msg': "Input should be 'carrier_a', 'carrier_b', 'carrier_c' or 'carrier_d'", 'input': 'carrier_x', 'ctx': {'expected': "'carrier_a', 'carrier_b', 'carrier_c' or 'carrier_d'"}}, {'type': 'greater_than', 'loc': ['body', 'weight_kg'], 'msg': 'Input should be greater than 0', 'input'

## How to read the outputs

The summary table is the posterior of the cost model. Carrier intercepts near 1.6, 1.4, 1.2 and 2.1 recover the simulation. Zone effects near 0 / 0.25 / 0.60 are metro / regional / remote. Weight and distance slopes near 0.46 and 0.20, noise near 0.25. R-hat 1.00 and ESS in the thousands means rerunning would not move a quote at the precision we publish. `zone_eff[metro]` is exactly zero by construction, so its R-hat is `nan` and its ESS is the full draw count.

PPC coverage near 0.90 means about 90% of training costs fell inside their own 90% predictive intervals. That is calibration on the training data, not a substitute for a later week (notebook 07).

The quote JSON:

- `expected_cost_usd` above `median_cost_usd` is the log-normal mean, not two competing models.
- `ci90_low_usd` / `ci90_high_usd` is a predictive interval: in 9 of 10 shipments like this one, the cost lands in that range. It is wider than a credible interval on the mean cost.
- `prob_cost_exceeds_quote` is the share of predictive draws above the price already shown to the customer.
- `out_of_training_range` on a 45 kg parcel means the weight is outside the training min/max. The service still returns a number and flags the extrapolation.
- `latency_ms` is compute inside the handler. There is no MCMC in it. A reference service answered in about 1 ms once warm.
- HTTP 422 lists the contract failures (unknown carrier, non-positive weight) and does not run the model.

Repeated calls differ in the cents because each request draws fresh observation noise from a shared generator. Seed per request if a client needs identical answers.

A reference quote for carrier B, regional, 2.5 kg, 600 km was about $9.50 expected, $9.13 median, 90% interval about $6.37 to $13.72, and about a 44% chance of exceeding a $9.50 quote. Read the JSON above the same way. The posterior draws move the cents across machines. The data hash in `/health` identifies the training file.

The Dockerfile builds a slim image with FastAPI, uvicorn, NumPy and pydantic only. Docker is not installed in this environment, so the image was not built. The uvicorn process above is the runtime that image starts.

## Pitfalls

- Running MCMC on the request path.
- Returning only a point estimate, so finance cannot see the interval.
- Training features that do not exist at request time.
- A new carrier with no policy: this contract returns 422. A pooled-prior fallback is a product decision, not a default.
- Business rules buried in the model service, so a margin change requires a refit.


## How to explain this to a CTO


In [5]:
body = normal.json()
print(
    f"For this parcel the expected shipping cost is ${body['expected_cost_usd']:.2f}. "
    f"In 9 of 10 shipments like it, the cost lands between "
    f"${body['ci90_low_usd']:.2f} and ${body['ci90_high_usd']:.2f}. "
    f"The chance the cost exceeds a ${9.50:.2f} quote is {body['prob_cost_exceeds_quote']:.0%}."
)
print(
    "Checkout can show one number. If undercharging costs more than overcharging, "
    "show a higher quantile than the mean, and keep the interval in the API so finance can use it. "
    f"The 45 kg request is outside the weights we trained on "
    f"(flag={heavy.json()['out_of_training_range']}), so a person should see that flag rather than a silent extrapolation."
)

For this parcel the expected shipping cost is $9.49. In 9 of 10 shipments like it, the cost lands between $6.33 and $13.58. The chance the cost exceeds a $9.50 quote is 44%.
Checkout can show one number. If undercharging costs more than overcharging, show a higher quantile than the mean, and keep the interval in the API so finance can use it. The 45 kg request is outside the weights we trained on (flag=True), so a person should see that flag rather than a silent extrapolation.


## Check yourself

**How do you serve a Bayesian model with low latency?**
Fit offline, thin the posterior to about 1,000 draws, ship the draws and a metadata file, and compute the posterior predictive with vectorised NumPy. For a heavier predictive, precompute a grid and interpolate. Never sample the posterior on the request.

**What is in the data contract?**
Inputs: names, types, units (kg, km, USD), ranges, enums, and which fields exist at request time. Outputs: a point, an interval with its level, probabilities of the business events, model name and version, and flags such as `out_of_training_range`. Pydantic models are the schema. A contract change bumps `/v1` to `/v2`.
